# Phase 3 Execution Notebook

Ian Solberg

March 9, 2026

Professor Richeng Piao

---

 **execution only notebook** 
 
 `data_handling.py` contains data decisions and useful functions for simplifying model assignments*

In [8]:
import pandas as pd
import numpy as np 
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from data_handling import MrozHandler
import statsmodels.api as sm
import statsmodels.formula.api as smf

norm = stats.norm

In [9]:
Mroz = MrozHandler("data/Mroz.csv")

Full dataset shape
Number of rows: 753
Number of features: 22
Working subset shape:
Number of rows: 428
Number of features: 22


#### Step 3.1: The Baseline Model (OLS Implementation)

##### step 3.1a - attach IMR vector as discussed in proposal 

In [10]:
# calculate IMR (Probability vector of choosing to work) using Probit:

Mroz.set_dependent("work", full=True)
Mroz.add_independents("kidslt6", "nwifeinc", "educ", "agew", "motheduc", "fatheduc", full=True)
probit_result = sm.Probit(Mroz.get_y().astype(int), Mroz.get_X()).fit()
fitted = probit_result.fittedvalues
imr_values = norm.pdf(fitted) / norm.cdf(fitted)
imr_series = pd.Series(imr_values, index=Mroz.full.index)


Mroz.attach("IMR", imr_series, to_working=True) # attach result
Mroz.clear_caches()

Dependent variable set to: work
Independent variables: ['kidslt6', 'nwifeinc', 'educ', 'agew', 'motheduc', 'fatheduc']
Optimization terminated successfully.
         Current function value: 0.603822
         Iterations 5
Attached 'IMR' to dataset
Caches cleared


In [11]:
def heckman_object(Mroz: MrozHandler) -> tuple[pd.Series, pd.DataFrame, MrozHandler]:
    Mroz.clear_caches()
    Mroz.set_dependent("lwage", full=False)
    Mroz.add_independents("exper", "expersq", full=False)
    Mroz.add_controls("educ", "kidslt6", "nwifeinc", "IMR", full=False)
    y = Mroz.get_y().dropna() # Fix the log wage issue by dropping nans, which correspond to non-working individuals
    X = Mroz.get_X().loc[y.index] # Ensure X and y are aligned after dropping nans
    return y, X, Mroz

y, X, Mroz = heckman_object(Mroz)

print(f"Dependent variable (y): {y.name}, shape: {y.shape}")
print(f"Independent variables (X): {X.columns.tolist()}, shape: {X.shape}")

Caches cleared
Dependent variable set to: lwage
Independent variables: ['exper', 'expersq']
Control variables: ['educ', 'kidslt6', 'nwifeinc', 'IMR']
Dependent variable (y): lwage, shape: (428,)
Independent variables (X): ['const', 'exper', 'expersq', 'educ', 'kidslt6', 'nwifeinc', 'IMR'], shape: (428, 7)


In [13]:
# Manual Implementation of OLS with Heteroskedasticity-Robust Errors

formula_1 = 'lwage ~ exper + expersq + educ + kidslt6 + nwifeinc + IMR'

model_1 = smf.ols(formula=formula_1, data=Mroz.full).fit(cov_type='HC1')

# Print the "Regression Anatomy"
print(model_1.summary())

                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.163
Model:                            OLS   Adj. R-squared:                  0.151
Method:                 Least Squares   F-statistic:                     14.55
Date:                Wed, 18 Mar 2026   Prob (F-statistic):           4.15e-15
Time:                        13:46:14   Log-Likelihood:                -429.95
No. Observations:                 428   AIC:                             873.9
Df Residuals:                     421   BIC:                             902.3
Df Model:                           6                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.4342      0.413     -1.053      0.2

#### Step 3.2: Testing Heterogeneity (Interaction Terms)

In [ ]:
formula_interact = 'lwage ~ exper + expersq + educ + educ:exper + kidslt6 + nwifeinc + IMR' # Do experience and education complement each other or act independently? 

model_interact = smf.ols(formula=formula_interact, data=Mroz.full).fit(cov_type='HC1')
print(model_interact.summary())

                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.164
Model:                            OLS   Adj. R-squared:                  0.150
Method:                 Least Squares   F-statistic:                     14.25
Date:                Wed, 18 Mar 2026   Prob (F-statistic):           1.24e-16
Time:                        13:48:48   Log-Likelihood:                -429.82
No. Observations:                 428   AIC:                             875.6
Df Residuals:                     420   BIC:                             908.1
Df Model:                           7                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.2963      0.510     -0.581      0.5

#### Step 3.3: The "Identification Police" Meeting

Before writing your final report, you must clear your model with the instructor or TA. This is a 10-minute "defense" of your strategy.

Required Preparation for Meeting:

The "Story": Can you explain the economic mechanism in plain English?

The Bias Check: Identify one variable you wish you had but don't (Omitted Variable Bias).

The Interpretation: Be ready to interpret the coefficient magnitude. (e.g., "A 1 year increase in education is associated with an 8.2% increase in wages, holding experience constant.")


----

#### Response:

# Phase 3: Model Specification & Causal Strategy

## 3.1 — The "Story": Economic Mechanism

We estimate a **Heckman selection-corrected Mincer wage equation** for married women using the 1975 Mroz dataset. The core economic question is: *what are the returns to education and experience for married women's wages, after accounting for the non-random selection into the labor force?*

The selection problem is straightforward, we only observe wages for women who choose to work. If the decision to work is correlated with unobserved determinants of wages (e.g., motivation, ability), then estimating a wage equation on working women alone produces biased coefficients. We address this via **Heckman's two-step procedure**:

1. **Selection equation (Probit):** We model labor force participation (`work`) as a function of `kidslt6`, `nwifeinc`, `educ`, `agew`, `motheduc`, and `fatheduc`. The parental education variables (`motheduc`, `fatheduc`) serve as **exclusion restrictions** — they plausibly affect the decision to work (through socialization and family norms) but do not directly determine wages.
2. **Outcome equation (OLS):** We regress log wages (`lwage`) on `exper`, `expersq`, `educ`, `kidslt6`, `nwifeinc`, and the **Inverse Mills Ratio (IMR)** computed from the selection equation. The IMR controls for selection bias.

The economic mechanism in the outcome equation reflects **human capital theory**: education and experience are investments that increase worker productivity, and therefore wages. The quadratic in experience (`expersq`) captures **diminishing returns**, early years of experience yield large wage gains, but the marginal return declines over time.

---

## 3.2 — Baseline Model: Coefficient Interpretations

Since the dependent variable is `lwage` (log of wages), all coefficients have a **semi-elasticity** interpretation, they represent approximate percentage changes in wages for a one-unit change in the regressor.

## Significant

| Variable | Coefficient | Std. Err. (HC1) | p-value | Interpretation |
|---|---|---|---|---|
| `educ` | 0.0983 | 0.030 | 0.001 | A one-year increase in education is associated with an approximately **9.8% increase in wages**, ceteris paribus. This is our most policy-relevant finding. |
| `exper` | 0.0249 | 0.007 | 0.001 | An additional year of experience is associated with approximately a **2.5% increase in wages** at the margin, holding education, family structure, and selection constant. |
| `expersq` | −0.0008 | 0.000 | 0.054 | The negative sign confirms **diminishing returns to experience**. The marginal return to experience declines as experience accumulates. Marginally significant at the 10% level. |

## Not Significant

| Variable | Coefficient | Std. Err. (HC1) | p-value | Interpretation |
|---|---|---|---|---|
| `kidslt6` | −0.0333 | 0.151 | 0.825 | Not statistically significant. Among working women, the presence of young children does not appear to affect wage levels. Note: young children likely affect the *decision* to work (captured in the selection equation) rather than the wage conditional on working. |
| `nwifeinc` | 0.0079 | 0.006 | 0.219 | Not statistically significant. Non-wife household income does not have a detectable association with wages among working women. |
| `IMR` | −0.0138 | 0.288 | 0.962 | Not statistically significant. We find **no strong evidence of selection bias** in this specification. This suggests either that selection into the labor force is approximately random with respect to unobserved wage determinants, or that our exclusion restrictions lack sufficient power to detect it. |

---

## 3.3 — Heterogeneity Test: Education × Experience Interaction

We test whether education and experience exhibit **human capital complementarity** — that is, whether more experienced women extract a higher wage return from each additional year of education.

**Specification:** `lwage ~ exper + expersq + educ + educ:exper + kidslt6 + nwifeinc + IMR`

**Result:** The interaction term `educ:exper` is small and insignificant (β = 0.0009, p = 0.577). We find **no evidence of complementarity** between education and experience in wage determination. The returns to education and experience appear to operate as **independent, additive channels**, consistent with a standard additive Mincer specification.

This null result is economically informative: a one-year increase in education yields approximately the same percentage wage premium regardless of whether the woman has 2 or 20 years of experience. The baseline (non-interacted) model is therefore our preferred parsimonious specification.

---

## 3.4 — Omitted Variable Bias Defense

The most concerning omitted variable in any Mincer wage equation is **innate ability or cognitive skill**. This variable satisfies both conditions for OVB:

1. **Correlated with the key regressor (education):** Higher-ability individuals are more likely to pursue additional years of schooling.
2. **Correlated with the outcome (wages):** Higher-ability individuals are likely to earn more, independent of their formal education.

Since ability is positively correlated with both education and wages, its omission likely produces an **upward bias** in our education coefficient. The true causal return to education is plausibly *smaller* than our estimate of 9.8%. This is a well-known limitation of cross-sectional Mincer equations resolving it would require instrumental variables (e.g., proximity to college, compulsory schooling laws) or twin/sibling fixed effects, which are unavailable in this dataset.

A secondary omitted variable is **geographic location** (urban vs. rural). Urban labor markets typically offer higher wages and may also be associated with higher educational attainment, potentially biasing both the education and experience coefficients upward. We could have potentially used the city binary variable to inform this decision better.

---

## Phase 3 Constraint Checklist

- [x] Ran a baseline regression using `statsmodels` (NOT sklearn)
- [x] Used Heteroskedasticity-Robust Standard Errors (`HC1`)
- [x] Included at least one Interaction Term to test for heterogeneity (`educ:exper`)
- [x] Interpreted the coefficients in a markdown cell (using "Ceteris Paribus" language)
- [x] Prepared "Omitted Variable Bias" defense argument (ability bias, geographic location)